In [1]:
!pip install -q transformers tokenizers datasets

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import (TensorDataset,
                              DataLoader)

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

import os
import math
import pickle

from tqdm import tqdm

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
tokenizer = Tokenizer.from_file(r"/content/drive/MyDrive/transformer/transformer-2/tokenizer.json")

In [5]:
with open(r"/content/drive/MyDrive/transformer/transformer-2/tokens.pkl","rb") as f:
  token = pickle.load(f)

In [6]:
data = torch.tensor(token,dtype=torch.long)
torch.save(data,r"/content/drive/MyDrive/transformer/checkpoints/data_ids.pt")

In [7]:
data.shape

torch.Size([434608])

In [8]:
VOCAB_SIZE = tokenizer.get_vocab_size()
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
BLOCK_SIZE = 128
BATCH_SIZE = 32
LR = 3e-4
EPOCHS = 100
SAVE_EVERY = 100

In [9]:
class Attention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        self.n_heads = n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        head_dim = C // self.n_heads

        q = q.view(B, T, self.n_heads, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, head_dim).transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(head_dim)
        scores = scores.masked_fill(mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        out = self.proj(out)
        return out


class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = Attention(d_model, n_heads)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len):
        super().__init__()

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.blocks = nn.ModuleList(
            [Block(d_model, n_heads) for _ in range(n_layers)]
        )

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = torch.triu(
            torch.ones(T, T, device=idx.device, dtype=torch.bool),
            diagonal=1
        )

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

In [10]:
model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    max_len=BLOCK_SIZE
).to(device)

param = sum(p.numel() for p in model.parameters())
print("Total params:", param)

Total params: 8489728


In [11]:
optimizer_function = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01
)

In [12]:
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer_function,
    T_max=EPOCHS
)

In [13]:
train_loss_hist = []
test_loss_hist = []

best_loss = float("inf")

patience = 0

progress_bar = tqdm(range(0, EPOCHS), colour="green")

for epoch in progress_bar:

    model.train()
    total_train_loss = 0
    train_batches = 0

    for _ in range(200):   # har epoch mein 200 batches

        starts = torch.randint(
            0,
            len(data) - BLOCK_SIZE - 1,
            (BATCH_SIZE,)
        )

        batch_x = torch.stack([data[s:s + BLOCK_SIZE] for s in starts]).to(device)
        batch_y = torch.stack([data[s + 1:s + BLOCK_SIZE + 1] for s in starts]).to(device)

        optimizer_function.zero_grad()

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            train_prediction = model(batch_x)
            train_loss = F.cross_entropy(
                train_prediction.view(-1, train_prediction.size(-1)),
                batch_y.view(-1)
            )

        scaler.scale(train_loss).backward()
        scaler.unscale_(optimizer=optimizer_function)

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer=optimizer_function)
        scaler.update()

        total_train_loss += train_loss.item()
        train_batches += 1

    avg_train_loss = total_train_loss / train_batches

    model.eval()
    total_test_loss = 0
    test_batches = 0

    with torch.no_grad():
        for _ in range(50):   # test ke liye 50 batches

            starts = torch.randint(
                0,
                len(data) - BLOCK_SIZE - 1,
                (BATCH_SIZE,)
            )

            batch_test_x = torch.stack([data[s:s + BLOCK_SIZE] for s in starts]).to(device)
            batch_test_y = torch.stack([data[s + 1:s + BLOCK_SIZE + 1] for s in starts]).to(device)

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                test_prediction = model(batch_test_x)
                test_loss = F.cross_entropy(
                    test_prediction.view(-1, test_prediction.size(-1)),
                    batch_test_y.view(-1)
                )

            total_test_loss += test_loss.item()
            test_batches += 1

    avg_test_loss = total_test_loss / test_batches

    scheduler.step()

    current_lr = optimizer_function.param_groups[0]["lr"]

    train_loss_hist.append(avg_train_loss)
    test_loss_hist.append(avg_test_loss)

    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer_function.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "test_loss": avg_test_loss,
        "best_loss": best_loss,
        "patience": patience,
    }, f"/content/drive/MyDrive/transformer/checkpoints/latest.pth")

    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer_function.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "test_loss": avg_test_loss,
    }, f"/content/drive/MyDrive/transformer/checkpoints/checkpoint_{epoch+1}.pth")

    if avg_test_loss < best_loss:

        best_loss = avg_test_loss
        patience = 0

        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer_function.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_loss": best_loss,
            "epoch": epoch,
        }, f"/content/drive/MyDrive/transformer/checkpoints/best_model.pth")

    else:
        patience += 1

    progress_bar.set_postfix({
        "epoch": epoch + 1,
        "Train Loss": f"{avg_train_loss:.4f}",
        "Test Loss": f"{avg_test_loss:.4f}",
        "LR": f"{current_lr:.6f}",
        "Best Loss": f"{best_loss:.4f}",
    })

    if patience >= 5:
        print("Early Stopping Triggered......")
        break

print("Model Has Trained+++++++++")

 86%|████████▌ | 86/100 [12:46<02:04,  8.91s/it, epoch=86, Train Loss=1.6650, Test Loss=1.6785, LR=0.000014, Best Loss=1.6585]


KeyboardInterrupt: 

In [ ]:
start_epoch = 0
best_loss = float("inf")
patience = 0

latest_path = f"/content/drive/MyDrive/transformer/checkpoints/latest.pth"

if os.path.exists(latest_path):
    print("Checkpoint mila, resume kar rahe hain...")

    checkpoint = torch.load(latest_path, map_location=device)

    model.load_state_dict(checkpoint["model"])
    optimizer_function.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])

    start_epoch = checkpoint["epoch"] + 1
    best_loss = checkpoint["best_loss"]
    patience = checkpoint["patience"]

    print(f"Resuming from epoch {start_epoch}, best loss {best_loss:.4f}")
else:
    print("Fresh training shuru")

In [18]:
prompt = "she is beautiful"

ids = tokenizer.encode(prompt).ids
x = torch.tensor([ids], dtype=torch.long).to(device)

with torch.no_grad():
    for _ in range(200):

        x_cond = x[:, -BLOCK_SIZE:]

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(x_cond)

        logits = logits[:, -1, :] / 0.8
        probs = F.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

print(tokenizer.decode(x[0].tolist()))

she is beautiful . " Thank you " of your magic ?" she is proud of you !" Tom and L ily nod ded . They eat El ly safely . They are happy . " We have to go on rides at the slide again ," the bald man said . " Maybe they can save it . They do it for a long time ." They decide to go to the man . They run and jump and climb up . They are almost and start to watch the ground . They go home . The fair were strong . They were happy ! L ily and Tom were playing a big structure . Lo ped to the kitchen and hid . They fell and dreamed of the house . Their mom and dad ran to the kitchen . They fell down stairs and hid behind the wall . They made a loud noise . " Look at the sky is round and co zy , it ' s no ." Tom says . He puts a lick their cars and books . " Anna and Ben ,
